In [ ]:
# Instalação do RFdiffusion3 via rc-foundry
!pip install -q "rc-foundry[rfd3]"
# RFD3 currently calls the AtomWorks 2.x parser API. Newer 3.x releases
# reject the extra_fields/build_assembly keywords during input parsing.
!pip install -q --no-deps "atomworks==2.2.1"
!foundry install rfd3 --checkpoint-dir /kaggle/working/models/rfd3

# Clone do ProteinMPNN
!test -d ProteinMPNN || git clone https://github.com/dauparas/ProteinMPNN.git

# Instalação do AlphaFold via ColabFold
!pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"

# Instalação de bibliotecas auxiliares para processamento de PDBs
!pip install -q biopython


In [ ]:
import os
import glob
import json
import subprocess
from pathlib import Path
import pandas as pd
from Bio import SeqIO
from Bio.PDB import PDBParser, MMCIFParser, PDBIO
import gzip
import warnings
from Bio import BiopythonWarning
from Bio.Seq import Seq

# Ignora warnings do PDBParser
warnings.simplefilter('ignore', BiopythonWarning)

# ==========================================
# 1. CONFIGURAÇÕES
# ==========================================
DATASET_PATH = "/kaggle/input/datasets/calistu/idr-core-hotspots" 
WORK_DIR = Path("/kaggle/working")
MPNN_DIR = WORK_DIR / "ProteinMPNN"
CKPT_PATH = WORK_DIR / "models/rfd3/rfd3_latest.ckpt"

VARIANT = "p36r" 
N_REFINEMENTS = 3
INITIAL_PARTIAL_T = 1
DECAY_RATE = 0.5

# ==========================================
# 2. FUNÇÕES DO PIPELINE
# ==========================================
def get_pdb_chains_info(pdb_path):
    """Lê um PDB e retorna um dicionário com o (início, fim) de cada cadeia."""
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("struct", str(pdb_path))
    chains_info = {}
    for model in struct:
        for chain in model:
            res_nums = [res.id[1] for res in chain if res.id[0] == ' ']
            if res_nums:
                chains_info[chain.id] = (res_nums[0], res_nums[-1])
        break # Considera apenas o primeiro modelo
    return chains_info

def parse_hotspots_dict(csv_path, top_n=5, chain="A"):
    """Extrai os hotspots e formata no padrão dicionário aplicando filtros de região"""
    df = pd.read_csv(csv_path, sep=";")
    valid_hotspots = []
    
    for _, row in df.iterrows():
        core_res = row['core_residue']
        
        # Filtro 1: Apenas resíduos do core > 200
        if core_res <= 200:
            continue
            
        # Extrai os parceiros do IDR como uma lista de inteiros
        partners_str = str(row['idr_partners'])
        if pd.isna(partners_str) or not partners_str.strip():
            continue
            
        partners = [int(p) for p in partners_str.split(',') if p.strip().isdigit()]
        if not partners:
            continue
            
        # Filtro 2: A maioria dos contatos no IDR deve estar no intervalo 30-60
        in_range_count = sum(1 for p in partners if 30 <= p <= 60)
        
        if in_range_count > len(partners) / 2:
            valid_hotspots.append(core_res)
            
        # Interrompe ao atingir o limite desejado (top_n)
        if len(valid_hotspots) == top_n:
            break

    return {f"{chain}{h}": "CA" for h in valid_hotspots}


def convert_rfd3_structure_to_pdb(structure_path, out_dir):
    """Convert an RFD3 mmCIF output to PDB for legacy ProteinMPNN."""
    structure_path = Path(structure_path)
    if structure_path.suffix.lower() == ".pdb":
        return structure_path

    parse_path = structure_path
    temporary_cif = None
    if structure_path.name.endswith(".cif.gz"):
        temporary_cif = Path(out_dir) / f".{structure_path.stem}.cif"
        with gzip.open(structure_path, "rb") as source, open(temporary_cif, "wb") as target:
            target.write(source.read())
        parse_path = temporary_cif

    if parse_path.suffix.lower() != ".cif":
        raise ValueError(f"Formato de estrutura RFD3 não suportado: {structure_path}")

    output_path = Path(out_dir) / f"{structure_path.name.replace('.cif.gz', '').replace('.cif', '')}.pdb"
    structure = MMCIFParser(QUIET=True).get_structure("rfd3", str(parse_path))
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(output_path))
    if temporary_cif is not None:
        temporary_cif.unlink(missing_ok=True)
    return output_path


def run_rfd3_dialect2(input_pdb, hotspots_dict, out_dir, task_name_prefix, partial_t=None, num_designs=10):
    """Gera o JSON no Dialect 2 multiplicando as tasks e executa o RFdiffusion3."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    chains_info = get_pdb_chains_info(input_pdb)
    target_start, target_end = chains_info.get('A', (1, 100))
    alvo_contexto = f"A{target_start}-{target_end}"
    
    # Criamos um dicionário vazio que vai guardar todas as "tarefas"
    round_config = {}
    
    for i in range(num_designs):
        task_name = f"{task_name_prefix}_model_{i}"
        
        if partial_t is None:
            # RODADA INICIAL
            task_dict = {
                "dialect": 2,
                "input": str(input_pdb),
                "contig": f"{alvo_contexto},/0,50-60",
                "select_fixed_atoms": alvo_contexto,
                "is_non_loopy": True,
                "infer_ori_strategy": "hotspots",
                "cif_parser_args": { "add_missing_atoms": True },
                "select_hotspots": hotspots_dict
            }
        else:
            # RODADA DE REFINAMENTO
            chain_ids = list(chains_info.keys())
            if len(chain_ids) > 1:
                binder_chain = chain_ids[1] 
                b_start, b_end = chains_info[binder_chain]
                binder_cadeia = f"{binder_chain}{b_start}-{b_end}"
            else:
                binder_cadeia = "B1-60"
                
            task_dict = {
                "dialect": 2,
                "input": str(input_pdb),
                "contig": f"{alvo_contexto},/0,{binder_cadeia}",
                "partial_t": partial_t,
                "select_fixed_atoms": alvo_contexto,
                "infer_ori_strategy": "hotspots",
                "select_hotspots": hotspots_dict,
                "cif_parser_args": { "add_missing_atoms": True },
                "is_non_loopy": True
            }
            
        round_config[task_name] = task_dict
    
    # Salva o JSON com N entradas
    round_json_path = out_dir / f"rfd3_input_{task_name_prefix}.json"
    with open(round_json_path, "w") as f:
        json.dump(round_config, f, indent=4)

    cmd = [
        "bash",
        "-lc",
        f"PYTORCH_ALLOC_CONF=expandable_segments:True rfd3 design out_dir={str(out_dir)} inputs={round_json_path} ckpt_path={CKPT_PATH} diffusion_batch_size=1"
    ]
    
    print(f"Rodando RFD3 ({task_name_prefix}) com {num_designs} designs...")
    
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except subprocess.CalledProcessError as e:
        print("\n!!! O RFDIFFUSION3 QUEBROU. VEJA O LOG ABAIXO !!!\n")
        print(e.stdout)
        raise

    generated_structures = (
        sorted(glob.glob(f"{out_dir}/**/*.pdb", recursive=True))
        + sorted(glob.glob(f"{out_dir}/**/*.cif", recursive=True))
        + sorted(glob.glob(f"{out_dir}/**/*.cif.gz", recursive=True))
    )
    generated_structures = [
        path for path in generated_structures
        if Path(path).name != round_json_path.name
    ]
    if not generated_structures:
        raise RuntimeError(f"RFD3 terminou sem gerar estruturas em {out_dir}")

    generated_pdbs = []
    for structure_path in generated_structures:
        pdb_path = convert_rfd3_structure_to_pdb(structure_path, out_dir)
        if pdb_path not in generated_pdbs:
            generated_pdbs.append(pdb_path)
    return generated_pdbs[:3]

def run_proteinmpnn(pdb_list, out_dir, num_seqs=100, top_n=10):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    fastas_gerados = []
    
    for pdb in pdb_list:
        pdb = Path(pdb)
        cmd = [
            "python", str(MPNN_DIR / "protein_mpnn_run.py"),
            "--pdb_path", str(pdb),
            "--out_folder", str(out_dir),
            "--num_seq_per_target", str(num_seqs),
            "--sampling_temp", "0.1"
        ]
        subprocess.run(cmd, check=True)
        
        base_name = pdb.stem
        fa_file = out_dir / "seqs" / f"{base_name}.fa"
        
        if fa_file.exists():
            records = list(SeqIO.parse(fa_file, "fasta"))
            designs = records[1:] 
            designs.sort(key=lambda r: float(r.description.split("score=")[1].split(",")[0]))
            
            top_designs = designs[:top_n]
            for i, record in enumerate(top_designs):
                # Converte para string, substitui o caractere e devolve como objeto Seq
                sequencia_corrigida = str(record.seq).replace("/", ":")
                record.seq = Seq(sequencia_corrigida)
                
                out_fasta = out_dir / f"{base_name}_design_{i}.fasta"
                SeqIO.write(record, out_fasta, "fasta")
                fastas_gerados.append(str(out_fasta))
                
    return fastas_gerados

def run_alphafold_and_rank(fasta_dir, out_dir, top_n=3):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    cmd = [
        "colabfold_batch",
        str(fasta_dir), str(out_dir),
        "--num-recycle", "2",
        "--num-models", "1",
        "--model-type", "alphafold2_multimer_v3"
    ]
    subprocess.run(cmd, check=True)
    
    scores = []
    for json_file in glob.glob(f"{out_dir}/*_scores_*.json"):
        with open(json_file, 'r') as f:
            data = json.load(f)
            iptm = data.get("iptm", 0)
            plddt_list = data.get("plddt", [0])
            plddt = sum(plddt_list) / len(plddt_list)
            combined_score = 0.8 * iptm + 0.2 * (plddt / 100)
            
            # Substitui '_scores_' por '_unrelaxed_' e '.json' por '.pdb'
            pdb_file = json_file.replace("_scores_", "_unrelaxed_")
            pdb_file = pdb_file[:-5] + ".pdb"  # remove .json e adiciona .pdb
            
            # Fallback caso tenha sido executado com relaxamento AMBER
            if not os.path.exists(pdb_file):
                pdb_relaxed = json_file.replace("_scores_", "_relaxed_")
                pdb_relaxed = pdb_relaxed[:-5] + ".pdb"
                if os.path.exists(pdb_relaxed):
                    pdb_file = pdb_relaxed

            if os.path.exists(pdb_file):
                scores.append((combined_score, pdb_file))
            else:
                print(f"[AVISO] PDB correspondente não encontrado para: {json_file}")
            
    scores.sort(key=lambda x: x[0], reverse=True)
    melhores_pdbs = [s[1] for s in scores[:top_n]]
    print(f"Top {top_n} AF2 Scores: {[round(s[0], 3) for s in scores[:top_n]]}")
    return melhores_pdbs

# ==========================================
# 3. EXECUÇÃO DO PIPELINE
# ==========================================
core_pdb = f"{DATASET_PATH}/{VARIANT}_representative_core.pdb"
hotspots_csv = f"{DATASET_PATH}/{VARIANT}_core_hotspots.csv"

print(f"--- INICIANDO PIPELINE PARA {VARIANT.upper()} ---")

hotspots_dict = parse_hotspots_dict(hotspots_csv, top_n=8)
print(f"Hotspots: {hotspots_dict}")

# --- RODADA INICIAL ---
print("\n[RODADA INICIAL] Gerando backbones...")
rfd3_out_dir = WORK_DIR / "results" / VARIANT / "round_0" / "rfd3"
top3_rfd3 = run_rfd3_dialect2(
    input_pdb=core_pdb, 
    hotspots_dict=hotspots_dict, 
    out_dir=rfd3_out_dir, 
    task_name_prefix=f"{VARIANT}_binder_design_0",  # ATUALIZADO AQUI
    num_designs=10
)

print("\n[RODADA INICIAL] ProteinMPNN...")
mpnn_out_dir = WORK_DIR / "results" / VARIANT / "round_0" / "mpnn"
fastas_for_af2 = run_proteinmpnn(top3_rfd3, mpnn_out_dir, num_seqs=100, top_n=10)

print("\n[RODADA INICIAL] AlphaFold...")
af2_out_dir = WORK_DIR / "results" / VARIANT / "round_0" / "af2"
best_af2_pdbs = run_alphafold_and_rank(mpnn_out_dir, af2_out_dir, top_n=3)

# --- REFINAMENTOS ITERATIVOS ---
current_best_pdbs = best_af2_pdbs
current_partial_t = INITIAL_PARTIAL_T

for i in range(1, N_REFINEMENTS + 1):
    print(f"\n{'='*40}")
    print(f"REFINAMENTO - RODADA {i} | partial_t: {current_partial_t:.2f}")
    
    round_rfd3_pdbs = []
    print("[REFINAMENTO] Ajustando backbones...")
    rfd3_out_dir = WORK_DIR / "results" / VARIANT / f"round_{i}" / "rfd3"
    
    for idx, af2_pdb in enumerate(current_best_pdbs):
        task_prefix = f"{VARIANT}_binder_step2_round_{i:02d}_mod_{idx}"
        pdbs_gerados = run_rfd3_dialect2(
            input_pdb=af2_pdb, 
            hotspots_dict=hotspots_dict, 
            out_dir=rfd3_out_dir, 
            task_name_prefix=task_prefix,  # ATUALIZADO AQUI
            partial_t=current_partial_t,
            num_designs=10
        )
        round_rfd3_pdbs.extend(pdbs_gerados)
    
    top3_refined = round_rfd3_pdbs[:3] 
    
    print("[REFINAMENTO] ProteinMPNN...")
    mpnn_out_dir = WORK_DIR / "results" / VARIANT / f"round_{i}" / "mpnn"
    fastas_for_af2 = run_proteinmpnn(top3_refined, mpnn_out_dir, num_seqs=100, top_n=5)
    
    print("[REFINAMENTO] AlphaFold...")
    af2_out_dir = WORK_DIR / "results" / VARIANT / f"round_{i}" / "af2"
    current_best_pdbs = run_alphafold_and_rank(mpnn_out_dir, af2_out_dir, top_n=3)
    
    current_partial_t -= DECAY_RATE
    
print("\nPipeline finalizado! Melhores binders salvos em:")
for b in current_best_pdbs:
    print(b)
